# WTA Match Predictor — EDA y Feature Engineering
Pipeline completo: carga, limpieza, clasificación de torneos, cálculo de features derivadas y generación del dataset de entrenamiento.

## 1. Imports y carga de datos

In [1]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport

c:\Users\NaiaJon\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\NaiaJon\AppData\Local\Temp\ipykernel_12908\1081620318.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
# Cuidado con las rutas al copiarlo al gihub
wta = pd.read_csv(r'C:\Users\NaiaJon\Documents\Naia\BootcampDataScience\Datos ML\wta.csv', low_memory=False)
print(f'Partidos cargados: {len(wta)}')
wta.head()

Partidos cargados: 44446


,Tournament,Date,Court,Surface,Round,Best of,Player_1,Player_2,Winner,Rank_1,Rank_2,Pts_1,Pts_2,Odd_1,Odd_2,Score
0,ASB Classic,2007-01-01 00:00:00,Outdoor,Hard,1st Round,3,Sun T.T.,Baker L.,Sun T.T.,81,272,332,90,1.33,3,6-1 6-1
1,ASB Classic,2007-01-01 00:00:00,Outdoor,Hard,1st Round,3,Myskina A.,Dulko G.,Dulko G.,16,59,1000,401,1.22,3.75,1-6 7-6 2-6
2,ASB Classic,2007-01-01 00:00:00,Outdoor,Hard,1st Round,3,Loit E.,Birnerova E.,Loit E.,56,84,418,324,1.72,2,6-1 6-1
3,ASB Classic,2007-01-01 00:00:00,Outdoor,Hard,1st Round,3,Nakamura A.,Craybas J.,Craybas J.,57,70,405,365,1.83,1.83,5-7 2-6
4,ASB Classic,2007-01-01 00:00:00,Outdoor,Hard,1st Round,3,Bartoli M.,Morita A.,Bartoli M.,18,180,951,152,1.16,4.5,7-6 6-3


## 2. Exploración inicial

In [3]:
wta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44446 entries, 0 to 44445
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Tournament  44446 non-null  object
 1   Date        44446 non-null  object
 2   Court       44446 non-null  object
 3   Surface     44446 non-null  object
 4   Round       44446 non-null  object
 5   Best of     44446 non-null  int64 
 6   Player_1    44446 non-null  object
 7   Player_2    44446 non-null  object
 8   Winner      44446 non-null  object
 9   Rank_1      44446 non-null  int64 
 10  Rank_2      44446 non-null  int64 
 11  Pts_1       44446 non-null  int64 
 12  Pts_2       44446 non-null  int64 
 13  Odd_1       44446 non-null  object
 14  Odd_2       44446 non-null  object
 15  Score       44446 non-null  object
dtypes: int64(5), object(11)
memory usage: 5.4+ MB


In [ ]:
# Profiling completo — genera wta_report.html
profile = ProfileReport(wta, title='WTA Profiling Report')
profile.to_file('wta_report.html')

## 3. Limpieza y transformaciones

In [ ]:
# Eliminar columnas no útiles para el modelo
# Court: muy desbalanceado (40k outdoor vs 400 indoor), la superficie ya lo captura
# Best of: en WTA siempre es al mejor de 3 sets
wta = wta.drop(columns=['Court', 'Best of'])

In [ ]:
# Unificar superficies: Greenset y Carpet tienen muy pocas entradas y son pistas rápidas → Hard
wta['Surface'] = wta['Surface'].replace({'Greenset': 'Hard', 'Carpet': 'Hard'})
print(wta['Surface'].value_counts())

In [ ]:
# Clasificar torneos por categoría usando palabras clave
# Los nombres cambian con patrocinadores pero la ciudad/torneo es estable

def clasificar_torneo(nombre):
    n = nombre.lower()
    
    if any(x in n for x in ['australian open', 'roland garros', 'french open',
                              'wimbledon', 'us open']):
        return 'GS'
    
    if any(x in n for x in ['wta finals', 'wta elite trophy', 'championships',
                              'tournament of champions', 'riyadh finals', 'wta tour championships']):
        return 'WTA_Finals'
    
    if any(x in n for x in ['indian wells', 'miami', 'madrid', 'roma', 'rome',
                              'internazionali', 'canada', 'toronto', 'rogers',
                              'canadian', 'cincinnati', 'beijing', 'china open',
                              'wuhan', 'doha', 'qatar', 'dubai', 'bnp paribas open',
                              'national bank open', 'sony ericsson open',
                              'western & southern financial group', 'shenzhen']):
        return 'WTA1000'
    
    if any(x in n for x in ['abu dhabi', 'abu dabi', 'adelaide', 'berlin', 'bett1open',
                              'eastbourne', 'rothesay', 'bad homburg', 'san jose',
                              'silicon valley', 'guangzhou', 'osaka', 'tokyo',
                              'toray pan pacific', 'seoul', 'strasbourg', 'birmingham',
                              'nottingham', 'brisbane', 'linz', 'merida', 'monterrey',
                              'charleston', 'stuttgart', 'porsche', 'washington',
                              'mubadala', 'guadalajara', 'gdl open akron',
                              'family circle cup', 'aegon', 'kremlin', 'pilot pen',
                              'new haven', 'bank of the west', 'sydney international',
                              'stanford', 'luxembourg', 'belgium', 'diamond games',
                              'fortis', 'san diego', 'zhengzhou', 'german']):
        return 'WTA500'
    
    return 'WTA250_o_menor'

wta['tournament_type'] = wta['Tournament'].map(clasificar_torneo)
print(wta['tournament_type'].value_counts())

In [ ]:
# Comprobación: torneos sin clasificar
sin_clasificar = wta[wta['tournament_type'] == 'WTA250_o_menor']['Tournament'].nunique()
print(f'Torneos únicos sin clasificar: {sin_clasificar}')

In [ ]:
# Eliminar Tournament (ya tenemos tournament_type) y convertir tipos
wta = wta.drop(columns='Tournament')

wta['Date']  = pd.to_datetime(wta['Date'], errors='coerce')
wta['Odd_1'] = pd.to_numeric(wta['Odd_1'], errors='coerce')
wta['Odd_2'] = pd.to_numeric(wta['Odd_2'], errors='coerce')

print('Nulos tras conversión:')
print(wta[['Date', 'Odd_1', 'Odd_2']].isna().sum())

In [ ]:
# Eliminar partidos sin fecha (3 registros con valor -1)
wta = wta.dropna(subset=['Date'])

In [ ]:
# Limpiar odds: valores < 1 son códigos de sin datos (equivalente a -1)
# Los convertimos a NaN — se usarán para comparar con casas pero no para entrenar
wta.loc[wta['Odd_1'] < 1, 'Odd_1'] = None
wta.loc[wta['Odd_2'] < 1, 'Odd_2'] = None

In [ ]:
# Calcular probabilidades implícitas de las casas de apuestas
# 1/odd y normalizar para que sumen 1 (elimina el margen de la casa)
wta['prob_1'] = 1 / wta['Odd_1']
wta['prob_2'] = 1 / wta['Odd_2']
total = wta['prob_1'] + wta['prob_2']
wta['prob_1'] = wta['prob_1'] / total
wta['prob_2'] = wta['prob_2'] / total

print(wta['prob_1'].describe())

In [ ]:
# Convertir tipos de columnas categóricas y de texto
wta['Surface']  = wta['Surface'].astype(str)
wta['Round']    = wta['Round'].astype(str)
wta['Score']    = wta['Score'].astype(str)
wta['Player_1'] = wta['Player_1'].astype(str)
wta['Player_2'] = wta['Player_2'].astype(str)
wta['Winner']   = wta['Winner'].astype(str)

In [ ]:
# Crear target: 1 si gana Player_1, 2 si gana Player_2
def asignar_target(fila):
    if fila['Winner'] == fila['Player_1']:
        return 1
    else:
        return 2

wta['target'] = wta.apply(asignar_target, axis=1)
print(wta['target'].value_counts())

In [ ]:
# Guardar wta limpio para uso en app.py y consultas
wta.to_csv('wta_limpio.csv', index=False)
print('wta_limpio.csv guardado')

## 4. Feature Engineering

Calculamos features derivadas para cada partido usando solo información disponible **antes** de ese partido (corte temporal estricto para evitar data leakage).

In [ ]:
def forma_reciente(df, jugadora, fecha_limite, meses=2):
    """Win rate de una jugadora en los N meses anteriores a la fecha.
    Devuelve 0 si no tiene partidos (lesión o pausa larga)"""
    fecha_inicio = fecha_limite - pd.DateOffset(months=meses)
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite) &
        (df['Date'] >= fecha_inicio)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [ ]:
def winrate(df, jugadora, fecha_limite, superficie=None, ronda=None):
    """Win rate histórico de una jugadora, filtrable por superficie y/o ronda.
    Devuelve 0.4 si no tiene historial (ligeramente por debajo de neutro = novata en esa condición)"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    if superficie:
        mask &= (df['Surface'] == superficie)
    if ronda:
        mask &= (df['Round'] == ronda)
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.4
    victorias = (partidos['Winner'] == jugadora).sum()
    return victorias/len(partidos)

In [ ]:
def headtohead(df, p1, p2, fecha_limite):
    """% de victorias de p1 sobre p2 en enfrentamientos directos previos a la fecha.
    Devuelve 0.5 si nunca se han enfrentado (neutro)"""
    mask = (
        (
            ((df['Player_1'] == p1) & (df['Player_2'] == p2)) |
            ((df['Player_1'] == p2) & (df['Player_2'] == p1))
        ) & (df['Date'] < fecha_limite)
    )
    partidos = df[mask]
    if len(partidos) == 0:
        return 0.5
    victorias_p1 = (partidos['Winner'] == p1).sum()
    return victorias_p1/len(partidos)

In [ ]:
def experiencia(df, jugadora, fecha_limite):
    """Número total de partidos jugados por la jugadora antes de la fecha"""
    mask = (
        ((df['Player_1'] == jugadora) | (df['Player_2'] == jugadora)) &
        (df['Date'] < fecha_limite)
    )
    return df[mask].shape[0]

In [ ]:
# Comprobación de las funciones antes de ejecutar el bucle completo
jugadora = 'Sabalenka A.'
fecha_limite = pd.to_datetime('2025-12-17')
print(f"Experiencia: {experiencia(wta, jugadora, fecha_limite)} partidos")
print(f"H2H vs Badosa: {headtohead(wta, jugadora, 'Badosa P.', fecha_limite):.2%}")
print(f"Win rate Clay: {winrate(wta, jugadora, fecha_limite, superficie='Clay'):.2%}")
print(f"Forma reciente: {forma_reciente(wta, jugadora, fecha_limite):.2%}")

### Construcción del dataset de features

⚠️ Este bucle tarda ~30 minutos. El resultado se guarda en CSV para no tener que repetirlo.

In [ ]:
# Reset index para usar como match_id
wta = wta.reset_index()

features = []
for i, partido in wta.iterrows():
    p1    = partido['Player_1']
    p2    = partido['Player_2']
    fecha = partido['Date']
    
    row = {
        # Identificador y metadata
        'match_id': i,
        'date':     partido['Date'],
        'odd_1':    partido['prob_1'],
        'odd_2':    partido['prob_2'],

        # Features del partido
        'surface':         partido['Surface'],
        'round':           partido['Round'],
        'tournament_type': partido['tournament_type'],

        # Features calculadas (solo info anterior al partido)
        'rank_diff':           float(partido['Rank_1']) - float(partido['Rank_2']),
        'wins2meses_p1':       forma_reciente(wta, p1, fecha),
        'wins2meses_p2':       forma_reciente(wta, p2, fecha),
        'ratio_superficie_p1': winrate(wta, p1, fecha, superficie=partido['Surface']),
        'ratio_superficie_p2': winrate(wta, p2, fecha, superficie=partido['Surface']),
        'h2h':                 headtohead(wta, p1, p2, fecha),
        'ratio_ronda_p1':      winrate(wta, p1, fecha, ronda=partido['Round']),
        'ratio_ronda_p2':      winrate(wta, p2, fecha, ronda=partido['Round']),
        'experiencia_p1':      experiencia(wta, p1, fecha),
        'experiencia_p2':      experiencia(wta, p2, fecha),

        # Target
        'target': partido['target']
    }
    features.append(row)

historico_partidos = pd.DataFrame(features)

# Feature de inexperiencia
historico_partidos['is_new_p1'] = (historico_partidos['experiencia_p1'] < 10).astype(int)
historico_partidos['is_new_p2'] = (historico_partidos['experiencia_p2'] < 10).astype(int)

print(f'Dataset construido: {len(historico_partidos)} partidos')
print(historico_partidos.info())

## 5. Limpieza del dataset de features

In [ ]:
# Eliminar 2007: primer año del dataset, las features derivadas están muy incompletas
# porque no hay historial previo para calcularlas
historico_partidos = historico_partidos[historico_partidos['date'].dt.year != 2007]
print(f'Partidos tras eliminar 2007: {len(historico_partidos)}')

In [ ]:
# Verificar nulos
print(historico_partidos.isnull().sum())

In [ ]:
# Guardar — carga posterior con pd.read_csv('historico_partidos.csv', parse_dates=['date'])
historico_partidos.to_csv('historico_partidos.csv', index=False)
print('✓ historico_partidos.csv guardado')

# Clustering. Perfiles de jugadoras

In [2]:
stats_2024 = pd.read_csv(r'C:\Users\NaiaJon\Documents\Naia\BootcampDataScience\Datos ML\wta_matches_2024.csv', low_memory=False)

In [3]:
stats_2024.columns

Index(['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level',
       'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry',
       'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
       'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand',
       'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round',
       'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
       'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt',
       'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced',
       'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points'],
      dtype='object')

In [4]:
stats_2024.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2024-9900,United Cup,Hard,18,I,20240101,299,216347,NaN,NaN,...,27.0,13.0,7.0,7.0,4.0,8.0,1.0,9505.0,NaN,NaN
1,2024-9900,United Cup,Hard,18,I,20240101,297,216347,NaN,NaN,...,47.0,28.0,7.0,12.0,3.0,8.0,1.0,9505.0,20.0,2330.0
2,2024-9900,United Cup,Hard,18,I,20240101,295,201493,NaN,NaN,...,82.0,47.0,10.0,15.0,11.0,19.0,NaN,NaN,292.0,246.0
3,2024-9900,United Cup,Hard,18,I,20240101,293,216347,NaN,NaN,...,27.0,18.0,6.0,8.0,6.0,11.0,1.0,9505.0,14.0,2770.0
4,2024-9900,United Cup,Hard,18,I,20240101,291,201614,NaN,NaN,...,72.0,51.0,22.0,16.0,3.0,6.0,20.0,2330.0,544.0,89.0
